# Pathway-map preparation — transparency report

This notebook documents **exactly how the Escher pathway maps and the gene crosswalk
were built** for the iCHO3K flux/expression explorer, with the data sources, the
identifier bridges, the coverage/reliability numbers, and worked examples.

**Why this matters.** The maps overlay predicted flux / gene expression onto reactions.
For that to be trustworthy, every reaction id on a map must correspond to the right
iCHO3K reaction, and every gene in the expression matrix must reach the right model
gene. Neither is automatic — iCHO3K, KEGG, RECON1, Ensembl and the RNA-seq data all use
different identifier namespaces. This report shows how each gap was bridged and how well.

**Contents**
1. The model and its namespaces
2. Gene crosswalk — Ensembl PICRH → Entrez (BioMart)
3. Reaction crosswalk — KEGG ↔ BiGG ↔ MetaNetX → iCHO3K
4. Map sources & the two map families (RECON1 curated, KEGG-derived)
5. Reliability scoring
6. The catalogue (what you actually get)
7. Worked example — render a map with flux overlay
8. Data artifacts & reproducibility
9. Known limitations

In [1]:
# Make `gems` importable whether or not it's pip-installed in this kernel.
# (Best: run Jupyter from the project venv — see README.)
import sys, pathlib
for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_c / 'src' / 'gems').is_dir():
        sys.path.insert(0, str(_c / 'src')); break
else:
    raise RuntimeError('Could not locate src/gems — run this notebook from inside the gems repo.')

In [2]:
import json, re
from collections import defaultdict, Counter
import pandas as pd
from gems import config as C, model_setup

model = model_setup.get_base_model()
print('iCHO3K base model:', len(model.reactions), 'reactions,',
      len(model.metabolites), 'metabolites,', len(model.genes), 'genes')

iCHO3K base model: 8368 reactions, 4695 metabolites, 2929 genes


## 1. The model and its namespaces

The base model is **iCHO3K** (LewisLabUCSD), CHO-S biomass, unblocked build. Two facts
drive everything downstream:

* **Reactions** use standard **BiGG** identifiers (`PFK`, `CSm`, `G6PDH2r`, …). iCHO3K is
  Recon3D-derived, so cytosolic/mitochondrial variants carry a compartment suffix
  (`CS` vs `CSm`).
* **Genes** are **NCBI Entrez** ids, mostly from the older *CriGri* annotation
  (the `100.7M` block). There are **no** KEGG/EC annotations on reactions and no gene
  symbols on genes — so every bridge has to be built from external cross-references.

In [3]:
genes = [g.id for g in model.genes]
blocks = Counter('100.7M (old CriGri)' if g.isdigit() and 100_700_000 <= int(g) <= 100_800_000
                 else '103M (newer)' if g.isdigit() and 103_000_000 <= int(g) <= 103_300_000
                 else 'other' for g in genes)
print('Entrez gene id blocks in iCHO3K:', dict(blocks))
print('example reaction ids:', [r.id for r in list(model.reactions)[:8]])

Entrez gene id blocks in iCHO3K: {'100.7M (old CriGri)': 2720, 'other': 173, '103M (newer)': 36}
example reaction ids: ['AGTim', 'AGTix', 'ARGSL', 'ARGSS', 'ASNN', 'ASNNm', 'ASNS1', 'ASPNATm']


## 2. Gene crosswalk — Ensembl PICRH → Entrez (BioMart)

The real DESeq2 counts are keyed by **Ensembl PICRH 1.0** gene ids (`ENSCGRG…`,
`CriGri-PICRH-1.0`). To score reactions we must reach the model's **Entrez** genes.

**Action taken.** Queried Ensembl BioMart (dataset `cgpicr_gene_ensembl`) for
`ensembl_gene_id → entrezgene_id` and wrote `data/maps/ensembl_picrh_to_entrez.tsv`.

```
# BioMart XML query (via https://www.ensembl.org/biomart/martservice)
<Query virtualSchemaName="default" formatter="TSV">
  <Dataset name="cgpicr_gene_ensembl">
    <Attribute name="ensembl_gene_id"/>
    <Attribute name="entrezgene_id"/>
  </Dataset>
</Query>
```

BioMart's PICRH annotation happens to include the **old CriGri Entrez ids** that iCHO3K
uses, so the mapping lands directly on the model's gene namespace.

In [4]:
from gems import gene_mapping
xwalk = gene_mapping.load_crosswalk()   # data/maps/ensembl_picrh_to_entrez.tsv
print(f'crosswalk rows      : {len(xwalk)}')
print(f'distinct Ensembl    : {xwalk.ensembl_gene_id.nunique()}')
print(f'distinct Entrez     : {xwalk.entrez_id.nunique()}')

model_genes = {g.id for g in model.genes}
reachable = model_genes & set(xwalk.entrez_id.astype(str))
print(f'\niCHO3K model genes  : {len(model_genes)}')
print(f'reachable via crosswalk: {len(reachable)}  = {len(reachable)/len(model_genes):.1%}  (target > 80%)')

crosswalk rows      : 21992
distinct Ensembl    : 21034
distinct Entrez     : 21274

iCHO3K model genes  : 2929
reachable via crosswalk: 2800  = 95.6%  (target > 80%)


In [5]:
# End-to-end check through the *actual* pipeline code path (what the run manifest reports).
# Simulate a DESeq2 matrix that contains every PICRH gene, then map one sample.
import numpy as np
ens = sorted(xwalk.ensembl_gene_id.unique())
counts = pd.DataFrame({'demo': np.random.default_rng(0).lognormal(4, 1, len(ens))}, index=ens)
scores, rep = gene_mapping.map_expression(model, counts, xwalk, 'demo')
rep.as_dict()   # 'model_gene_coverage' is what lands in results/fluxes/manifest.json

{'n_expressed_genes': 21034,
 'n_mapped_to_entrez': 21034,
 'entrez_coverage': 1.0,
 'n_model_genes': 2929,
 'n_model_genes_with_data': 2800,
 'model_gene_coverage': 0.956,
 'n_reactions_scored': 6068}

> **Result: model-gene coverage ≈ 95%** — comfortably above the 0.8 target. When real
> Ensembl-keyed counts are dropped into `data/raw/`, the pipeline reports this same
> `mapping_coverage` block in `results/fluxes/manifest.json`.
>
> *Note:* the GSE140669 demonstration dataset is keyed by NCBI **symbol/LOC**, not
> Ensembl, so it uses a separate crosswalk (`gse140669_symbol_to_entrez.tsv`, ~66%
> coverage) — pass it with `--crosswalk`. The BioMart file above is the crosswalk for
> the real PICRH-keyed project data.

## 2b. Master gene-ID bridge (all namespaces in one table)

`scripts/build_gene_id_master.py` builds **`data/maps/gene_id_master.tsv`** — one row per
Chinese-hamster gene linking every identifier namespace that shows up across CHO
transcriptomics + modelling:

`ensembl_gene_id · entrez_id · symbol · refseq_mrna · refseq_protein · ensembl_protein ·
uniprot · gene_biotype · chromosome · in_icho3k · icho3k_n_reactions · description`

Sources: Ensembl BioMart (`cgpicr_gene_ensembl`, PICRH-1.0) for Ensembl/Entrez/RefSeq/
UniProt/symbol, joined to the iCHO3K model (Entrez) + its symbol map. **All 2,929 model
genes are present** (2,793 with a PICRH Ensembl id; 136 appended without one).

`gems.gene_ids` is the accessor: `translate()`, `crosswalk()`, `detect_namespace()`,
and `make_best_crosswalk()` (auto-detect a dataset's namespace → build the entrez
crosswalk). This means a dataset keyed by **symbol, RefSeq, Ensembl or UniProt** can all
reach the model through one bridge.

In [6]:
from gems import gene_ids as gid
master = gid.load_master()
print('master rows:', len(master), '| columns:', list(master.columns))
gid.summary().to_frame('non_null')

master rows: 34795 | columns: ['ensembl_gene_id', 'entrez_id', 'symbol', 'gene_biotype', 'chromosome', 'refseq_mrna', 'refseq_protein', 'ensembl_protein', 'uniprot', 'in_icho3k', 'icho3k_n_reactions', 'description']


,non_null
ensembl_gene_id,34659
entrez_id,22045
symbol,23057
refseq_mrna,19525
refseq_protein,19418
ensembl_protein,24551
uniprot,15305
rows,34795
in_icho3k,2938


In [7]:
# how many iCHO3K model genes each namespace can reach (via the master)
mg = {g.id for g in model.genes}
cov = {ns: len(mg & set(gid.crosswalk(ns, 'entrez_id')['entrez_id'])) / len(mg)
       for ns in ['entrez_id','symbol','ensembl_gene_id','refseq_mrna','refseq_protein','uniprot']}
pd.Series(cov, name='model_gene_coverage').sort_values(ascending=False).map('{:.1%}'.format)

entrez_id          100.0%
symbol              97.1%
ensembl_gene_id     95.4%
refseq_mrna         95.2%
refseq_protein      95.2%
uniprot             74.5%
Name: model_gene_coverage, dtype: object

In [8]:
# example: fully-linked rows for model genes
cols = ['ensembl_gene_id','entrez_id','symbol','refseq_mrna','refseq_protein','uniprot','icho3k_n_reactions']
master[master.in_icho3k & master.ensembl_gene_id.notna() & master.refseq_mrna.notna()][cols].head(6)

,ensembl_gene_id,entrez_id,symbol,refseq_mrna,refseq_protein,uniprot,icho3k_n_reactions
1,ENSCGRG00015007570,100751055,Has2,XM_027431680,XP_027287481,A0A8C2LAN0,2
28,ENSCGRG00015004708,100766249,LOC100766249,XM_035441429;XM_027406090;XM_035441430,XP_027298289;XP_027261891;XP_035297321;XP_0352...,A0A061I9Z6,1
29,ENSCGRG00015006133,100764981,Ces4a,XM_035441442,XP_035297333,A0A061I734,1
40,ENSCGRG00015003858,100766785,Tk2,XM_027405735,XP_027261536,A0A061ICV1,7
46,ENSCGRG00015006067,100757439,Ces3,XM_027405770;XM_035442616;XM_035442617,XP_003508580;XP_027261571;XP_035315797;XP_0352...,A0A061IAF0;A0A9J7JWN2;A0A061ICK7,1
48,ENSCGRG00015004783,100765571,LOC100765571,XM_035441432;XM_027405759;XM_035441431,XP_027261560;XP_035297322;XP_035297323,A0A061IEJ2,1


## 3. Reaction crosswalk — KEGG ↔ BiGG ↔ MetaNetX → iCHO3K

KEGG gives us pathway **layout + connectivity**, but its reactions are `R#####` ids. To
colour them by iCHO3K flux we must map `KEGG reaction → iCHO3K BiGG reaction`. Two
bridges are unioned for recall:

1. **Direct** — BiGG's namespace lists `kegg.reaction` links for ~2.8k reactions.
2. **MetaNetX** — `reac_xref.tsv` maps *both* `kegg.reaction:R#####` and
   `bigg.reaction:ID` to the same MNX id (same version), recovering reactions BiGG
   doesn't link to KEGG directly.

Both are matched to iCHO3K by exact BiGG id or compartment-stripped base (`CS`↔`CSm`).

In [9]:
from gems import kegg_maps
kegg2model, src = kegg_maps.load_reaction_crosswalk(model)
print('KEGG reactions mapped to iCHO3K:', src['total'])
print('  of which recovered via MetaNetX bridge:', src['mnx_added'])
# ambiguity: KEGG reactions that map to >1 iCHO3K reaction (mostly compartment variants)
amb = sum(1 for v in kegg2model.values() if len(v) > 1)
print(f'  ambiguous (>1 iCHO3K match): {amb} ({amb/max(len(kegg2model),1):.0%})')

KEGG reactions mapped to iCHO3K: 915
  of which recovered via MetaNetX bridge: 255
  ambiguous (>1 iCHO3K match): 248 (27%)


## 4. Map sources & the two map families

| Family | Source | Layout | Ids | Reliability |
|---|---|---|---|---|
| **RECON1 curated** | escher.github.io | hand-drawn | BiGG (direct) | 🟢 High |
| **KEGG-derived** | KEGG **human (hsa)** KGML | hand-drawn (KEGG) | KEGG→BiGG→iCHO3K | 🟡/🔴 Medium/Low |

**Why human (hsa) and not the reference KGML?** The reference `rn` maps include every
organism's reactions, so glycolysis looked like *21%* coverage even though iCHO3K has the
whole pathway (the rest were bacterial/plant variants). Using **hsa** restricts the
denominator to human reactions, which iCHO3K (Recon3D-derived) largely contains — pushing
coverage to a meaningful level.

**Readability.** Metabolites/reactions carry human-readable names (KEGG compound list +
the model's reaction names), and currency metabolites (ATP/H2O/NAD…) are drawn as small
secondary nodes so the backbone stands out. Metabolite nodes are **shared by KGML entry**
id, so reactions wire into genuine connected cascades (not isolated reaction-cards).

## 5. Reliability scoring

Each map in `escher_maps/manifest.json` carries:
* `coverage` = mapped / total pathway reactions (headline metric),
* `n_mapped`, `n_reactions`,
* `ambiguous_frac` = fraction of mapped reactions with >1 iCHO3K match (a value was chosen),
* `tier`: **High** = curated RECON1; **Medium** = KEGG ≥ 40%; **Low** = KEGG < 40%.

Reactions on a map that aren't in iCHO3K render **grey** ("no data"), so the map never
silently invents flux.

In [10]:
manifest = json.loads((C.ROOT / 'escher_maps' / 'manifest.json').read_text())
rows = [{'category': cat, **it} for cat, items in manifest.items() for it in items]
df = pd.DataFrame(rows)
print('maps:', len(df), '| categories:', df.category.nunique())
print('tiers:', df.tier.value_counts().to_dict())
df.groupby('tier')['coverage'].describe()[['count', 'mean', 'min', 'max']]

maps: 55 | categories: 9
tiers: {'Medium': 40, 'Low': 10, 'High': 5}


,count,mean,min,max
tier,,,,
High,5.0,0.817000,0.733,0.884
Low,10.0,0.301100,0.106,0.393
Medium,40.0,0.635075,0.400,1.000


In [11]:
# top maps by coverage
df.sort_values('coverage', ascending=False)[
    ['category', 'pathway', 'tier', 'coverage', 'n_mapped', 'n_reactions', 'source']
].head(20).reset_index(drop=True)

,category,pathway,tier,coverage,n_mapped,n_reactions,source
0,Carbohydrate,Butanoate metabolism,Medium,1.000,14,14,KEGG hsa00650 (human) × iCHO3K
1,Cofactors & vitamins,Pantothenate and CoA biosynthesis,Medium,0.938,15,16,KEGG hsa00770 (human) × iCHO3K
2,Nucleotide,Pyrimidine metabolism,Medium,0.897,52,58,KEGG hsa00240 (human) × iCHO3K
3,Central carbon,RECON1.Glycolysis TCA PPP,High,0.884,38,43,RECON1 (curated)
4,Amino acid,Arginine biosynthesis,Medium,0.867,13,15,KEGG hsa00220 (human) × iCHO3K
5,Carbohydrate,RECON1.Carbohydrate metabolism,High,0.855,289,338,RECON1 (curated)
6,Lipid & vitamins,RECON1.Inositol retinol metabolism,High,0.821,96,117,RECON1 (curated)
7,Carbohydrate,Glyoxylate and dicarboxylate metabolism,Medium,0.800,20,25,KEGG hsa00630 (human) × iCHO3K
8,Amino acid,"Glycine, serine and threonine metabolism",Medium,0.794,27,34,KEGG hsa00260 (human) × iCHO3K
9,Amino acid,RECON1.Amino acid metabolism (partial),High,0.792,248,313,RECON1 (curated)


## 6. The catalogue by category

In [12]:
summary = (df.groupby('category')
             .agg(maps=('pathway', 'count'),
                  mean_coverage=('coverage', 'mean'),
                  high=('tier', lambda s: (s == 'High').sum()),
                  medium=('tier', lambda s: (s == 'Medium').sum()),
                  low=('tier', lambda s: (s == 'Low').sum()))
             .round(2).sort_values('maps', ascending=False))
summary

,maps,mean_coverage,high,medium,low
category,,,,,
Carbohydrate,17,0.59,1,14,2
Amino acid,14,0.65,2,10,2
Lipid,9,0.35,0,4,5
Cofactors & vitamins,8,0.61,0,7,1
Nucleotide,2,0.80,0,2,0
Other,2,0.69,0,2,0
Central carbon,1,0.88,1,0,0
Energy,1,0.62,0,1,0
Lipid & vitamins,1,0.82,1,0,0


## 7. Worked example — render a map with flux overlay

Load a real sample's flux vector and colour/size a map by it. `RECON1.Glycolysis TCA PPP`
(High, 88%) is the connected central-carbon cascade — glycolysis feeding the
mitochondrial TCA loop.

In [13]:
from gems import escher_view as ev
flux = pd.read_parquet(C.FLUX_DIR / 'flux_matrix.parquet'); flux.index = flux.index.astype(str)
sample = flux.columns[0]

map_path = C.ROOT / 'escher_maps' / 'glycolysis_tca_ppp.json'
emap = json.loads(map_path.read_text())
data = ev.reaction_data_for_sample(flux, sample)
print('sample:', sample)
print('map coverage on this sample:', f"{ev.coverage_for_map(emap, data):.0%}")

sample: 3936_ST
map coverage on this sample: 88%


In [14]:
# inline interactive Escher render (thickness = |flux|, colour = direction)
# NOTE: uses gems.escher_view.show_map (offline iframe), NOT escher.Builder --
# Builder's ipywidget frontend needs classic-notebook nbextensions and hangs in JupyterLab.
ev.show_map(map_path, data, mode='flux')

/Users/nkranjc/work/gems/.venv/lib/python3.9/site-packages/IPython/core/display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [15]:
# a KEGG-derived example with human-readable names
tca_map_path = C.ROOT / 'escher_maps' / 'kegg_hsa00020.json'
tca = json.loads(tca_map_path.read_text())
print('metabolite names:', [n['name'] for n in tca[1]['nodes'].values()
                            if n['node_type'] == 'metabolite'][:8])
print('reaction labels :', [r['name'] for r in list(tca[1]['reactions'].values())[:6]])

ev.show_map(tca_map_path, data, mode='flux')

metabolite names: ['Succinate', 'Succinyl-CoA', 'Acetyl-CoA', 'Oxaloacetate', 'Citrate', '(S)-Malate', 'Fumarate', 'cis-Aconitate']
reaction labels : ['Succinate--CoA ligase (ADP-forming)', 'Succinate--CoA ligase (GDP-forming)', 'ATP-Citrate lyase', 'fumarase', 'Aconitase (half-reaction B, Isocitrate hydro-lyase)', 'Aconitase (half-reaction A, Citrate hydro-lyase)']


## 8. Data artifacts & reproducibility

Everything is cached under `data/maps/` and rebuilt by `scripts/prepare_escher_maps.py`.

| Artifact | Source | Used for |
|---|---|---|
| `ensembl_picrh_to_entrez.tsv` | Ensembl BioMart `cgpicr_gene_ensembl` | gene → model Entrez |
| `bigg_models_reactions.txt` | bigg.ucsd.edu namespace | BiGG↔KEGG, BiGG↔MNX |
| `mnx_reac_xref.tsv` | metanetx.org `reac_xref` | KEGG↔BiGG via MetaNetX |
| `kegg_compound_names.txt` | rest.kegg.jp `list/compound` | readable metabolite names |
| `kgml/hsa*.xml` | rest.kegg.jp `get/<pid>/kgml` | pathway layout + connectivity |
| `escher_maps/*.json` + `manifest.json` | generated | the maps + reliability |

Rebuild everything:
```bash
python scripts/prepare_escher_maps.py    # maps + manifest (+ vendors escher.min.js)
streamlit run app/streamlit_app.py       # explore
```

In [16]:
for f in ['ensembl_picrh_to_entrez.tsv', 'bigg_models_reactions.txt', 'mnx_reac_xref.tsv',
          'kegg_compound_names.txt', 'gse140669_symbol_to_entrez.tsv']:
    p = C.DATA / 'maps' / f
    print(f'{"OK " if p.exists() else "-- "}{f:<34} {p.stat().st_size//1024 if p.exists() else 0:>7} KB')
print('cached KGML files:', len(list((C.DATA / 'maps' / 'kgml').glob('*.xml'))))

OK ensembl_picrh_to_entrez.tsv            794 KB
OK bigg_models_reactions.txt            10445 KB
OK mnx_reac_xref.tsv                    78859 KB
OK kegg_compound_names.txt                961 KB
OK gse140669_symbol_to_entrez.tsv         402 KB
cached KGML files: 242


## 9. Known limitations (read before trusting a map)

* **KEGG-derived maps show the connected core.** BiGG/MNX only bridge a subset of KEGG
  reactions, so peripheral/alternative reactions on a Medium/Low map may be grey even if
  iCHO3K technically has them. Trust the connected backbone; treat sparse edges cautiously.
* **Compartment ambiguity.** When one KEGG reaction maps to several iCHO3K reactions
  (e.g. cytosolic + mitochondrial), one is chosen deterministically. `ambiguous_frac`
  flags how often this happened per map.
* **Metabolite overlay is approximate on KEGG maps.** Metabolite ids come from a KEGG→BiGG
  base-id map for labelling; reaction overlay (the thing we colour) is the reliable part.
* **RECON1 maps are large** — they open zoomed-to-fit and look sparse until you zoom in.
* **Calibration caveat (upstream).** The flux values themselves depend on the placeholder
  medium / `FLUX_SCALE`; see the main README. This report is about *map identity/coverage*,
  not the biological calibration of the fluxes.